# Pillar 4: Context Managers & Resource Management

## Core Mechanics & Theory

Context managers (`with` blocks) **guarantee setup and teardown logic runs reliably**—even if exceptions occur.

In GPU programming, AI training, and systems code, they are used everywhere:

- Switching CUDA devices: `with torch.cuda.device(1):`
- Disabling autograd graphs: `with torch.no_grad():`
- GPU profiling/timing: `with torch.cuda.amp.autocast():`
- Synchronizing streams and acquiring thread/process locks

## The Core Principle

> **"Do something before the block, let the block run, and ALWAYS do cleanup afterward."**

## 1. Why Do We Even Need with?

### The Problem: Manual Resource Management

**Without context managers:**

```python
file = open("data.txt")
data = file.read()
file.close()
```

**What if something goes wrong?**

```python
file = open("data.txt")
data = file.read()
# 💥 exception happens here
file.close()  # ← Never runs!
```

If an exception happens before `file.close()`, you might not clean up properly.

### The Solution: Context Managers

```python
with open("data.txt") as file:
    data = file.read()
# Python GUARANTEES cleanup happens when you leave the with block
```

Python guarantees that the cleanup runs **no matter what**—even if an exception occurs.

## 2. The Context Manager Protocol

**The 4-Step Lifecycle:**

1. `obj = ContextObject()` — Create the context manager object
2. `target = obj.__enter__()` — Setup logic / Acquire resource
3. Execute inner code block — Your code runs here
4. `obj.__exit__(exc_type, exc_val, exc_tb)` — Teardown logic / Release resource (even if exception occurred)

## 3. Context Manager as a Lifecycle

When you write:

```python
with Something() as x:
    do_something()
```

Think of it as:

```
        Create object
             ↓
       __enter__()
             ↓
      ┌─────────────┐
      │             │
      │  your code  │
      │             │
      └─────────────┘
             ↓
        __exit__()
             ↓
          cleanup
```

**In Plain English:**

- **`__enter__()`** means: "We're entering the context. Prepare/acquire resources."
- **`__exit__()`** means: "We're leaving the context. Clean up/release resources."

## 4. The Functional Approach: @contextlib.contextmanager

Instead of writing a full class with `__enter__` and `__exit__`, you can write a **generator function** decorated with `@contextlib.contextmanager`:

**How it works:**

- **Everything before `yield`** is the setup (equivalent to `__enter__()`)
- **The value passed to `yield`** is bound to the `as target`
- **Everything after `yield`** (inside a `finally:` block) is the cleanup (equivalent to `__exit__()`)

## 5. Decorators vs. Context Managers

**Decorators** → Change/enhance a **function**
- The decorator says: "Whenever this function is called, do this extra stuff."

**Context Managers** → Manage what happens before/after a **code block**
- The context manager says: "Before this block starts, do this. When the block ends, do this."

## 6. The Biggest Difference: Visual Comparison

### Decorator Approach

```python
@timer
def calculate():
    step1()
    step2()
    step3()
```

The decorator **wraps the entire function** definition:

```
timer
  ↓
calculate()
  │
  ├── step1
  ├── step2
  └── step3
```

### Context Manager Approach

```python
with timer():
    step1()
    step2()
    step3()
```

The context manager **wraps the specific code block**:

```
timer setup
     ↓
 ┌───────────┐
 │ step1     │
 │ step2     │
 │ step3     │
 └───────────┘
     ↓
timer cleanup
```

**That's a huge distinction** — decorators wrap function definitions, context managers wrap execution blocks!

In [5]:
from contextlib import contextmanager
import time
@contextmanager
def timer():
    
    start = time.perf_counter()
    try:
        yield
    finally:
        print(start-time.perf_counter())

In [6]:
with timer():
    result = sum( i* i for i in range(10000000))

-0.8736856000032276
